In [1]:
import openml
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
import warnings

np.random.seed(42)
warnings.filterwarnings('ignore')


def estimate_features_after_ohe(data_id, task_type='classification', sample_size=None):
    """
    Estimate the number of features after one-hot encoding for an OpenML dataset.
    
    Args:
        data_id: OpenML dataset ID
        task_type: 'classification' or 'regression'
        sample_size: Number of rows to sample for estimation (None = use entire dataset)
        
    Returns:
        Estimated number of features after one-hot encoding
    """
    try:
        dataset = fetch_openml(data_id=data_id, as_frame=True)
        if dataset.target is not None:
            X = dataset.data.copy()
        else:
            if dataset.frame is None:
                return None
            X = dataset.frame.copy()
        
        if X.empty or X.shape[1] == 0:
            return None
        
        # Sample if requested (to speed up estimation)
        if sample_size is not None and len(X) > sample_size:
            X = X.sample(n=min(sample_size, len(X)), random_state=42)
        
        # Count features after one-hot encoding
        # Categorical columns (object type): one feature per unique value
        # Numerical columns: one feature each
        num_features = 0
        for col in X.columns:
            col_dtype = X[col].dtype
            if col_dtype.kind == 'O':  # Object/categorical
                # Count unique values (after one-hot encoding, each unique value becomes a feature)
                n_unique = X[col].nunique()
                num_features += n_unique
            elif col_dtype.name == 'category':  # Pandas category type
                n_unique = X[col].nunique()
                num_features += n_unique
            else:  # Numerical (int, float, bool)
                num_features += 1
        
        return num_features
    except Exception as e:
        return None


def sample_datasets(df, n_samples, random_state=42):
    """
    Perform simple random sampling from a DataFrame.
    
    Args:
        df: DataFrame to sample from (index should contain dataset IDs)
        n_samples: Number of samples to select
        random_state: Random seed
        
    Returns:
        DataFrame with selected samples (preserves original index with dataset IDs)
    """
    if len(df) <= n_samples:
        return df.copy()
    # Sample and preserve the original index (which contains dataset IDs)
    return df.sample(n=n_samples, random_state=random_state)


def compute_max_features_after_ohe(instances, max_complexity=2000000):
    """
    Compute maximum allowed features after OHE based on instance count.
    
    Training time scales roughly with instances × features, so we use a product threshold.
    Small datasets can have more features, large datasets should have fewer.
    
    Args:
        instances: Number of instances in the dataset
        max_complexity: Maximum allowed product (instances × features_after_ohe)
    
    Returns:
        Maximum allowed features after one-hot encoding
    """
    if instances <= 0:
        return 0
    max_features = max_complexity / instances

    return int(max_features)


def select_classification_datasets(n=10, max_instances=50000, max_features=100, 
                                   max_complexity=2000000, random_state=42):
    """
    Select binary classification datasets from CC-18 suite with adaptive filtering.
    
    Uses adaptive filtering: small datasets can have more features after OHE,
    large datasets must have fewer features to keep training time reasonable.
    
    Args:
        n: Number of datasets to select
        max_instances: Maximum number of instances
        max_features: Maximum number of raw features (before one-hot encoding)
        max_complexity: Maximum allowed product (instances × features_after_ohe)
        random_state: Random seed
        
    Returns:
        DataFrame with selected classification datasets, including 'FeaturesAfterOHE' 
        and 'MaxAllowedFeatures' columns
    """
    # Get CC-18 suite
    cc18 = openml.study.get_suite("OpenML-CC18")
    df_cc18 = openml.datasets.list_datasets(data_id=cc18.data, output_format='dataframe')
    
    # Filter: binary classification (2 classes), size constraints
    df_class = df_cc18[
        (df_cc18["NumberOfClasses"] == 2) &
        (df_cc18["NumberOfInstances"] <= max_instances) &
        (df_cc18["NumberOfFeatures"] <= max_features)
    ].copy()
    
    # Estimate features after one-hot encoding for each dataset
    print(f"Estimating features after one-hot encoding for {len(df_class)} datasets...")
    features_after_ohe = []
    max_allowed_features = []
    for i, (idx, row) in enumerate(df_class.iterrows(), 1):
        print(f"  [{i}/{len(df_class)}] Dataset {idx} ({row['name']}): ", end='', flush=True)
        n_features = estimate_features_after_ohe(idx, task_type='classification')
        features_after_ohe.append(n_features)
        
        # Compute max allowed features for this instance count
        n_instances = row['NumberOfInstances']
        max_allowed = compute_max_features_after_ohe(n_instances, max_complexity)
        max_allowed_features.append(max_allowed)
        
        if n_features is not None:
            status = "✅" if n_features <= max_allowed else "❌ (exceeds limit)"
            print(f"{n_features} features after OHE (raw: {row['NumberOfFeatures']}, "
                  f"max allowed: {max_allowed}) {status}")
        else:
            print("failed")
    
    df_class['FeaturesAfterOHE'] = features_after_ohe
    df_class['MaxAllowedFeatures'] = max_allowed_features
    
    # Filter by adaptive threshold: features_after_ohe <= max_allowed for this instance count
    df_class = df_class[
        (df_class['FeaturesAfterOHE'].notna()) &
        (df_class['FeaturesAfterOHE'] <= df_class['MaxAllowedFeatures'])
    ].copy()
    
    print(f"\nAfter adaptive filtering: {len(df_class)} datasets remain")
    if len(df_class) > 0:
        print(f"  Complexity range: {df_class['NumberOfInstances'] * df_class['FeaturesAfterOHE']}")
        print(f"  Min complexity: {(df_class['NumberOfInstances'] * df_class['FeaturesAfterOHE']).min():.0f}")
        print(f"  Max complexity: {(df_class['NumberOfInstances'] * df_class['FeaturesAfterOHE']).max():.0f}")
    
    # Simple random sampling
    return sample_datasets(df_class, n, random_state=random_state)


def select_regression_datasets(n=10, min_instances=1000, max_instances=50000, 
                               max_features=100, max_complexity=2000000, random_state=42):
    """
    Select regression datasets from CTR23 suite with adaptive filtering.
    
    Uses adaptive filtering: small datasets can have more features after OHE,
    large datasets must have fewer features to keep training time reasonable.
    
    Args:
        n: Number of datasets to select
        min_instances: Minimum number of instances
        max_instances: Maximum number of instances
        max_features: Maximum number of raw features (before one-hot encoding)
        max_complexity: Maximum allowed product (instances × features_after_ohe)
        random_state: Random seed
        
    Returns:
        DataFrame with selected regression datasets, including 'FeaturesAfterOHE'
        and 'MaxAllowedFeatures' columns
    """
    # Get CTR23 regression suite
    ctr23 = openml.study.get_suite(353)
    df_ctr23 = openml.datasets.list_datasets(data_id=ctr23.data, output_format='dataframe')
    
    # Filter: size constraints
    df_reg = df_ctr23[
        (df_ctr23["NumberOfInstances"] >= min_instances) &
        (df_ctr23["NumberOfInstances"] <= max_instances) &
        (df_ctr23["NumberOfFeatures"] <= max_features)
    ].copy()
    
    # Estimate features after one-hot encoding for each dataset
    print(f"Estimating features after one-hot encoding for {len(df_reg)} datasets...")
    features_after_ohe = []
    max_allowed_features = []
    for i, (idx, row) in enumerate(df_reg.iterrows(), 1):
        print(f"  [{i}/{len(df_reg)}] Dataset {idx} ({row['name']}): ", end='', flush=True)
        n_features = estimate_features_after_ohe(idx, task_type='regression')
        features_after_ohe.append(n_features)
        
        # Compute max allowed features for this instance count
        n_instances = row['NumberOfInstances']
        max_allowed = compute_max_features_after_ohe(n_instances, max_complexity)
        max_allowed_features.append(max_allowed)
        
        if n_features is not None:
            status = "✅" if n_features <= max_allowed else "❌ (exceeds limit)"
            print(f"{n_features} features after OHE (raw: {row['NumberOfFeatures']}, "
                  f"max allowed: {max_allowed}) {status}")
        else:
            print("failed")
    
    df_reg['FeaturesAfterOHE'] = features_after_ohe
    df_reg['MaxAllowedFeatures'] = max_allowed_features
    
    # Filter by adaptive threshold: features_after_ohe <= max_allowed for this instance count
    df_reg = df_reg[
        (df_reg['FeaturesAfterOHE'].notna()) &
        (df_reg['FeaturesAfterOHE'] <= df_reg['MaxAllowedFeatures'])
    ].copy()
    
    print(f"\nAfter adaptive filtering: {len(df_reg)} datasets remain")
    if len(df_reg) > 0:
        complexity = df_reg['NumberOfInstances'] * df_reg['FeaturesAfterOHE']
        print(f"  Complexity range: {complexity.min():.0f} - {complexity.max():.0f}")
    
    # Simple random sampling
    return sample_datasets(df_reg, n, random_state=random_state)


In [2]:
# Select classification datasets
# max_complexity controls training time: instances × features_after_ohe <= max_complexity
# Lower values = faster training, higher values = allow more complex datasets
classification_datasets = select_classification_datasets(
    n=10, 
    max_features=100,
    max_complexity=100000,
    max_instances=50000,
    random_state=42
)
classification_datasets[["name", "NumberOfInstances", "NumberOfFeatures", "FeaturesAfterOHE", "MaxAllowedFeatures", "NumberOfClasses"]]


Estimating features after one-hot encoding for 30 datasets...
  [1/30] Dataset 3 (kr-vs-kp): 73 features after OHE (raw: 37.0, max allowed: 31) ❌ (exceeds limit)
  [2/30] Dataset 15 (breast-w): 9 features after OHE (raw: 10.0, max allowed: 143) ✅
  [3/30] Dataset 29 (credit-approval): 46 features after OHE (raw: 16.0, max allowed: 144) ✅
  [4/30] Dataset 31 (credit-g): 61 features after OHE (raw: 21.0, max allowed: 100) ✅
  [5/30] Dataset 37 (diabetes): 8 features after OHE (raw: 9.0, max allowed: 130) ✅
  [6/30] Dataset 38 (sick): 53 features after OHE (raw: 30.0, max allowed: 26) ❌ (exceeds limit)
  [7/30] Dataset 44 (spambase): 57 features after OHE (raw: 58.0, max allowed: 21) ❌ (exceeds limit)
  [8/30] Dataset 50 (tic-tac-toe): 27 features after OHE (raw: 10.0, max allowed: 104) ✅
  [9/30] Dataset 151 (electricity): 14 features after OHE (raw: 9.0, max allowed: 2) ❌ (exceeds limit)
  [10/30] Dataset 1049 (pc4): 37 features after OHE (raw: 38.0, max allowed: 68) ✅
  [11/30] Dataset

,name,NumberOfInstances,NumberOfFeatures,FeaturesAfterOHE,MaxAllowedFeatures,NumberOfClasses
15,breast-w,699.0,10.0,9,143,2.0
23381,dresses-sales,500.0,13.0,156,200,2.0
1510,wdbc,569.0,31.0,30,175,2.0
29,credit-approval,690.0,16.0,46,144,2.0
1067,kc1,2109.0,22.0,21,47,2.0
1049,pc4,1458.0,38.0,37,68,2.0
1464,blood-transfusion-service-center,748.0,5.0,4,133,2.0
37,diabetes,768.0,9.0,8,130,2.0
40983,wilt,4839.0,6.0,5,20,2.0
6332,cylinder-bands,540.0,40.0,162,185,2.0


In [3]:
# Select regression datasets
# max_complexity controls training time: instances × features_after_ohe <= max_complexity
# Lower values = faster training, higher values = allow more complex datasets
regression_datasets = select_regression_datasets(
    n=10, 
    max_features=100,
    max_complexity=100000,
    max_instances=50000,
    random_state=42
)
regression_datasets[["name", "NumberOfInstances", "NumberOfFeatures", "FeaturesAfterOHE", "MaxAllowedFeatures"]]


Estimating features after one-hot encoding for 26 datasets...
  [1/26] Dataset 41021 (Moneyball): 72 features after OHE (raw: 15.0, max allowed: 81) ✅
  [2/26] Dataset 44956 (abalone): 10 features after OHE (raw: 9.0, max allowed: 23) ✅
  [3/26] Dataset 44957 (airfoil_self_noise): 5 features after OHE (raw: 6.0, max allowed: 66) ✅
  [4/26] Dataset 44958 (auction_verification): 16 features after OHE (raw: 8.0, max allowed: 48) ✅
  [5/26] Dataset 44959 (concrete_compressive_strength): 8 features after OHE (raw: 9.0, max allowed: 97) ✅
  [6/26] Dataset 44963 (physiochemical_protein): 9 features after OHE (raw: 10.0, max allowed: 2) ❌ (exceeds limit)
  [7/26] Dataset 44964 (superconductivity): 81 features after OHE (raw: 82.0, max allowed: 4) ❌ (exceeds limit)
  [8/26] Dataset 44966 (solar_flare): 29 features after OHE (raw: 11.0, max allowed: 93) ✅
  [9/26] Dataset 44969 (naval_propulsion_plant): 14 features after OHE (raw: 15.0, max allowed: 8) ❌ (exceeds limit)
  [10/26] Dataset 44971 (

,name,NumberOfInstances,NumberOfFeatures,FeaturesAfterOHE,MaxAllowedFeatures
44966,solar_flare,1066.0,11.0,29,93
41021,Moneyball,1232.0,15.0,72,81
44987,socmob,1156.0,6.0,39,86
45402,space_ga,3107.0,7.0,6,32
44957,airfoil_self_noise,1503.0,6.0,5,66
44956,abalone,4177.0,9.0,10,23
44980,kin8nm,8192.0,9.0,8,12
44959,concrete_compressive_strength,1030.0,9.0,8,97
44972,red_wine,1599.0,12.0,11,62
44958,auction_verification,2043.0,8.0,16,48


In [4]:
# Test timing for one split per dataset
import subprocess
import time
import tempfile
import os
import shutil
from pathlib import Path
from hp_tuning_utils import load_hp_tuning_conf

# Get dataset IDs
classification_ids = classification_datasets.index.tolist()
regression_ids = regression_datasets.index.tolist()

# Default hyperparameters for timing test
default_hp = {
    'dropout': 0.2,
    'l2_regularization': 1e-5,
    'learning_rate': 0.01,
    'feature_dropout': 0.05,
    'output_regularization': 0.05
}

# Load configs
class_config = Path('hp_tuning_conf/openml_classification.json')
reg_config = Path('hp_tuning_conf/openml_regression.json')
_, class_fixed = load_hp_tuning_conf(class_config)
_, reg_fixed = load_hp_tuning_conf(reg_config)

def run_timing_test(dataset_id, task_type, timeout_s=3600):
    """Run one split and measure time."""
    dataset_name = f"OpenML_{dataset_id}_{task_type}"
    fixed_hp = (class_fixed if task_type == 'classification' else reg_fixed).copy()
    fixed_hp['dataset_name'] = dataset_name
    fixed_hp['regression'] = (task_type == 'regression')
    
    # Temp logdir
    temp_logdir = os.path.join(tempfile.gettempdir(), f'nam_timing_{dataset_id}')
    os.makedirs(temp_logdir, exist_ok=True)
    
    # Build command
    all_hp = {**fixed_hp, **default_hp}
    all_hp['logdir'] = os.path.join(temp_logdir, 'trial_1')
    all_hp['data_split'] = 1
    
    cmd = ['python', '-m', 'neural_additive_models.nam_train']
    for k, v in all_hp.items():
        if v is None:
            cmd.append(f'--{k}=None')
        elif isinstance(v, bool):
            cmd.append(f'--{k}={"true" if v else "false"}')
        else:
            cmd.append(f'--{k}={v}')
    
    # Run
    print(f"{dataset_name:40s}...", end=' ', flush=True)
    start = time.time()
    try:
        result = subprocess.run(' '.join(cmd), shell=True, capture_output=True, 
                               text=True, timeout=timeout_s)
        elapsed = time.time() - start
        success = result.returncode == 0
        print(f"{'✅' if success else '❌'} {elapsed:.1f}s")
        return {'dataset_id': dataset_id, 'name': dataset_name, 'task': task_type,
                'time': elapsed, 'success': success}
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start
        print(f"⏱️ {elapsed:.1f}s (timeout)")
        return {'dataset_id': dataset_id, 'name': dataset_name, 'task': task_type,
                'time': elapsed, 'success': False}
    finally:
        shutil.rmtree(temp_logdir, ignore_errors=True)

# Run tests
print("="*70)
print("TIMING TESTS - One Split Per Dataset")
print("="*70)

results = []
print("\nClassification:")
for did in classification_ids:
    results.append(run_timing_test(did, 'classification'))

print("\nRegression:")
for did in regression_ids:
    results.append(run_timing_test(did, 'regression'))

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
df = pd.DataFrame(results)
print(df[['name', 'time', 'success']].to_string(index=False))
print(f"\nTotal time: {df['time'].sum():.1f}s")
print(f"Average: {df['time'].mean():.1f}s")
print(f"Successful: {df['success'].sum()}/{len(df)}")


Instructions for updating:
non-resource variables are not supported in the long term


TIMING TESTS - One Split Per Dataset

Classification:
OpenML_15_classification                ... ✅ 16.7s
OpenML_23381_classification             ... ✅ 200.3s
OpenML_1510_classification              ... ✅ 43.8s
OpenML_29_classification                ... ✅ 57.5s
OpenML_1067_classification              ... ✅ 37.1s
OpenML_1049_classification              ... ✅ 174.2s
OpenML_1464_classification              ... ✅ 15.0s
OpenML_37_classification                ... ✅ 21.4s
OpenML_40983_classification             ... ✅ 30.2s
OpenML_6332_classification              ... ✅ 329.4s

Regression:
OpenML_44966_regression                 ... ✅ 70.6s
OpenML_41021_regression                 ... ✅ 162.9s
OpenML_44987_regression                 ... ✅ 68.8s
OpenML_45402_regression                 ... ✅ 74.9s
OpenML_44957_regression                 ... ✅ 26.3s
OpenML_44956_regression                 ... ✅ 48.7s
OpenML_44980_regression                 ... ✅ 98.0s
OpenML_44959_regression                 ... ✅